# AI 기반 Linux 취약점 진단 - Google Colab GPU

기존 `ai_based.py`를 **Notebook 하나에서 직접 실행**하도록 변환한 버전입니다.

실행 흐름:

```text
GitHub Clone
    ↓
result.xml + knowledge/*.json 로드
    ↓
ITEM/@code ↔ knowledge["U-xxx"] 매핑
    ↓
REVIEW=true 항목 추출
    ↓
Qwen + Colab GPU
    ↓
AI 분석 결과 출력
```

별도의 `ai_based.py` 실행 없이 이 Notebook 자체가 전체 AI 분석 코드를 포함합니다.

## 1. GPU 확인

In [28]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab 메뉴에서 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요."
    )

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. 패키지 설치

In [29]:
!pip -q install -U transformers accelerate sentencepiece

## 3. GitHub 저장소 연결

본인의 GitHub 주소로 수정하세요.

권장 구조:

```text
repository/
├── result.xml
├── knowledge/
│   └── linux_knowledge.json
└── notebooks/
    └── ai_based_colab.ipynb
```

In [30]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/mandoo-hub/server_check-with_AI-.git"
BRANCH = "main"
REPO_DIR = Path("/content/security-assessment/content")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull"],
        check=True
    )

os.chdir(REPO_DIR)

RESULT_FILE = REPO_DIR / "result.xml"
KNOWLEDGE_DIR = REPO_DIR / "knowledge"

print("Repository :", REPO_DIR)
print("Result XML :", RESULT_FILE)
print("Knowledge  :", KNOWLEDGE_DIR)

Repository : /content/security-assessment/content
Result XML : /content/security-assessment/content/result.xml
Knowledge  : /content/security-assessment/content/knowledge


## 4. 파일 및 code 매핑 사전 확인

In [36]:
import json
import xml.etree.ElementTree as ET

if not RESULT_FILE.exists():
    raise FileNotFoundError(RESULT_FILE)

if not KNOWLEDGE_DIR.exists():
    raise FileNotFoundError(KNOWLEDGE_DIR)


# ============================================================
# Knowledge ID 수집
# ============================================================

knowledge_keys = set()

for jf in sorted(KNOWLEDGE_DIR.glob("*.json")):

    print(f"[+] Checking: {jf.name}")

    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("[DEBUG] type =", type(data).__name__)

    if isinstance(data, dict):

        print(
            "[DEBUG] top-level keys =",
            list(data.keys())[:20]
        )

        # ----------------------------------------
        # 단일 항목 JSON
        #
        # {
        #   "id": "U-001",
        #   ...
        # }
        # ----------------------------------------

        if "id" in data:

            code_value = str(
                data["id"]
            ).strip()

            knowledge_keys.add(
                code_value
            )

            print(
                f"[DEBUG] 등록된 Knowledge ID: "
                f"{code_value}"
            )


        # ----------------------------------------
        # 기존 code 형식도 지원
        # ----------------------------------------

        elif "code" in data:

            code_value = str(
                data["code"]
            ).strip()

            knowledge_keys.add(
                code_value
            )

            print(
                f"[DEBUG] 등록된 Knowledge ID: "
                f"{code_value}"
            )


        # ----------------------------------------
        # 여러 항목이 들어 있는 형식
        #
        # {
        #   "U-001": {...},
        #   "U-002": {...}
        # }
        # ----------------------------------------

        else:

            for key in data.keys():

                key = str(key).strip()

                if key.startswith("U-"):

                    knowledge_keys.add(
                        key
                    )

                    print(
                        f"[DEBUG] 등록된 Knowledge ID: "
                        f"{key}"
                    )


# ============================================================
# result.xml REVIEW=true 코드 수집
# ============================================================

root = ET.parse(
    RESULT_FILE
).getroot()

review_codes = []

for item in root.findall(".//ITEM"):

    review = (
        item.findtext(
            "./AI/REVIEW",
            "false"
        )
        .strip()
        .lower()
        == "true"
    )

    if not review:
        continue

    code_value = (
        item.get(
            "code",
            ""
        )
        .strip()
    )

    if code_value:
        review_codes.append(
            code_value
        )


# ============================================================
# 매핑 결과 확인
# ============================================================

print()
print(
    "Knowledge IDs :",
    sorted(knowledge_keys)
)

print(
    "Knowledge Items :",
    len(knowledge_keys)
)

print(
    "AI REVIEW Items :",
    len(review_codes)
)

print()


for code_value in review_codes:

    print(
        f"{code_value:7} "
        f"MATCH={code_value in knowledge_keys}"
    )


missing = [
    code_value
    for code_value in review_codes
    if code_value not in knowledge_keys
]


if missing:

    raise RuntimeError(
        "Knowledge 미매핑 코드: "
        + ", ".join(missing)
    )


print()
print(
    "[OK] 모든 REVIEW 항목의 code가 "
    "Knowledge와 매핑됩니다."
)

[+] Checking: knowledge.json
[DEBUG] type = dict
[DEBUG] top-level keys = ['id', 'guide_code', 'name', 'category', 'category_en', 'target_system', 'security_references', 'description', 'decision_logic', 'evidence_mapping', 'ai_guidance', 'reference']
[DEBUG] 등록된 Knowledge ID: U-001

Knowledge IDs : ['U-001']
Knowledge Items : 1
AI REVIEW Items : 1

U-001   MATCH=True

[OK] 모든 REVIEW 항목의 code가 Knowledge와 매핑됩니다.


## 5. AI 모델 및 분석 코드

아래 셀은 기존 `ai_based.py` 전체 로직을 포함합니다.

Colab용 변경점:
- GPU 자동 사용
- tokenizer tensor를 GPU로 이동
- `result.xml`의 `ITEM/@code` 사용
- Evidence를 `key = value [source] [note]` 구조로 전달
- `review_items.append()`를 FIELD 반복문 밖에서 실행

In [53]:
import json
import re
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# 설정
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 실제 /knowledge를 사용할 경우
KNOWLEDGE_DIR = Path("./knowledge")

# 프로젝트 내부 knowledge 폴더를 사용할 경우에는 아래처럼 변경
# KNOWLEDGE_DIR = Path(__file__).resolve().parent / "knowledge"


# ============================================================
# Knowledge Base 로드
# ============================================================

def load_knowledge(knowledge_dir=KNOWLEDGE_DIR):
    """
    /knowledge 디렉터리 내부의 모든 *.json 파일을 읽어
    하나의 knowledge dictionary로 병합한다.
    """

    knowledge_dir = Path(knowledge_dir)

    if not knowledge_dir.exists():
        raise FileNotFoundError(
            f"Knowledge 디렉터리가 존재하지 않습니다: {knowledge_dir}"
        )

    if not knowledge_dir.is_dir():
        raise NotADirectoryError(
            f"Knowledge 경로가 디렉터리가 아닙니다: {knowledge_dir}"
        )

    json_files = sorted(knowledge_dir.glob("*.json"))

    if not json_files:
        raise FileNotFoundError(
            f"{knowledge_dir} 디렉터리에 JSON 파일이 없습니다."
        )

    knowledge = {}

    print(f"[+] Knowledge Directory : {knowledge_dir}")
    print(f"[+] Knowledge Files     : {len(json_files)}")

    total = len(json_files)

    for index, json_file in enumerate(json_files, start=1):

        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # ------------------------------------------------
            # 형식 1
            #
            # {
            #     "U-01": {...}
            # }
            # ------------------------------------------------

            if isinstance(data, dict):

                # 단일 항목 파일
                # {
                #   "code": "U-01",
                #   "title": ...
                # }
                if "code" in data or "id" in data:


                    code = str(
                        data.get("code")
                        or data.get("id")
                    ).strip()

                    if code in knowledge:
                        print(
                            f"\n[!] Duplicate Knowledge ID: "
                            f"{code} ({json_file.name})"
                        )

                    knowledge[code] = data

                # 여러 항목이 들어 있는 파일
                else:
                    for code, item_data in data.items():

                        code = str(code).strip()

                        if code in knowledge:
                            print(
                                f"\n[!] Duplicate Knowledge ID: "
                                f"{code} ({json_file.name})"
                            )

                        knowledge[code] = item_data

            else:
                print(
                    f"\n[!] Skip invalid knowledge file: "
                    f"{json_file.name}"
                )

        except json.JSONDecodeError as e:
            print(
                f"\n[!] JSON Parsing Error "
                f"({json_file.name}): {e}"
            )

        except Exception as e:
            print(
                f"\n[!] Knowledge Load Error "
                f"({json_file.name}): {e}"
            )

        # 진행률 표시
        percent = (index / total) * 100

        print(
            f"\r[+] Loading Knowledge "
            f"[{index}/{total}] "
            f"{percent:6.2f}% "
            f"- {json_file.name}",
            end="",
            flush=True
        )

    print()
    print(f"[+] Loaded Knowledge Items : {len(knowledge)}")

    return knowledge


# ============================================================
# Local LLM 로드
# ============================================================

def load_model():

    print("[+] Loading Local LLM...")
    print(f"[+] Model : {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
            device_map="auto"
        )
        print(f"[+] GPU : {torch.cuda.get_device_name(0)}")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="cpu"
        )
        print("[!] CUDA unavailable - CPU mode")

    model.eval()

    print("[+] Local LLM Loaded")

    return tokenizer, model


# ============================================================
# Knowledge 검색
# ============================================================

def retrieve_knowledge(code, knowledge):

    return knowledge.get(code)


# ============================================================
# XML AI REVIEW 여부
# ============================================================

def is_ai_review_enabled(criteria):

    """
    criteria.xml의 다음 값을 확인

    <AI>
        <REVIEW>true</REVIEW>
    </AI>

    true인 경우 AI 분석 수행
    """

    value = criteria.findtext(
        "./AI/REVIEW",
        default="false"
    )

    return value.strip().lower() == "true"


# ============================================================
# AI 설정 가져오기
# ============================================================

def get_ai_config(criteria):

    ai = criteria.find("./AI")

    if ai is None:
        return {
            "recommended": False,
            "trigger_reason": "NONE",
            "review": False
        }

    recommended = (
        ai.findtext("RECOMMENDED", "false")
        .strip()
        .lower()
        == "true"
    )

    trigger_reason = (
        ai.findtext("TRIGGER_REASON", "NONE")
        .strip()
    )

    review = (
        ai.findtext("REVIEW", "false")
        .strip()
        .lower()
        == "true"
    )

    return {
        "recommended": recommended,
        "trigger_reason": trigger_reason,
        "review": review
    }


# ============================================================
# Prompt 생성
# ============================================================

def build_prompt(
    code,
    evidence,
    knowledge_item,
    trigger_reason="NONE"
):

    kb_text = json.dumps(
        knowledge_item,
        ensure_ascii=False,
        indent=2
    )

    item_name = (
        knowledge_item.get("name")
        or knowledge_item.get("title")
        or ""
    )

    if isinstance(evidence, list):
        evidence_text = "\n".join(
            str(x)
            for x in evidence
        )
    else:
        evidence_text = str(evidence)

    prompt = f"""
당신은 Linux 서버 보안 취약점 진단 전문가입니다.

제공된 보안 기준과 시스템 증적만 사용하여
해당 점검 항목을 분석하십시오.

[점검 항목]
Code: {code}
제목: {item_name}

[AI 분석 요청 사유]
{trigger_reason}

[보안 지식]
{kb_text}

[시스템 수집 증적]
{evidence_text}

[판단 규칙]

1. [시스템 수집 증적]에 제공된 모든 Evidence를 먼저 확인한다.

2. Knowledge의 good_criteria를 label 단위로 하나씩 평가한다.

3. 각 label 내부 conditions에는 condition_operator를 적용한다.
   - AND이면 필요한 모든 condition을 만족해야 PASS이다.
   - OR이면 하나 이상의 condition을 만족하면 PASS이다.

4. 각 label의 결과를 PASS, FAIL, UNKNOWN 중 하나로 결정한다.

5. 모든 good_criteria label 평가가 끝난 후에만
   decision_logic.operator를 적용한다.
   - OR이면 하나 이상의 label이 PASS일 경우 GOOD이다.
   - AND이면 모든 label이 PASS일 경우 GOOD이다.

6. vulnerable_criteria에 해당하는 Evidence가 있더라도,
   다른 good_criteria의 대체 통제가 PASS인지 반드시 확인한다.

7. 대체 통제가 PASS이면 특정 다른 통제가 FAIL이라는 이유만으로
   VULNERABLE을 반환해서는 안 된다.

8. 특정 Evidence 하나만 보고 최종 판단하지 않는다.

9. 필요한 Evidence가 부족하여 label을 확정할 수 없으면 UNKNOWN으로 판단한다.

10. 최종 결론을 낼 수 없으면 MANUAL로 판단한다.

11. result는 label_results 및 decision_expression과 반드시 논리적으로 일치해야 한다.

12. matched_evidence에는 실제 [시스템 수집 증적]에 존재하는 문자열만 사용한다.

출력 형식:

13. label_results에는 반드시 decision_logic.good_criteria에 정의된 label만 평가한다.

14. vulnerable_criteria의 label을 good_criteria와 동일한 논리식에 섞지 않는다.

15. decision_expression은 반드시 decision_logic.operator와
    good_criteria label의 PASS/FAIL/UNKNOWN 결과만 사용하여 작성한다.

16. Knowledge의 description, name, examples 문장을 Evidence로 사용해서는 안 된다.

17. label_results[].evidence와 matched_evidence에는
    반드시 [시스템 수집 증적]에 실제로 존재하는 key=value 형식의 문자열만 사용한다.

18. 시스템 Evidence에 존재하지 않는 문장을 Evidence로 생성하거나 요약해서는 안 된다.

{{
    "result": "GOOD | VULNERABLE | MANUAL",
    "confidence": 0.0,

    "label_results": [
        {{
            "label": "Knowledge의 실제 label 이름",
            "status": "PASS | FAIL | UNKNOWN",
            "evidence": [
                "실제 Evidence"
            ]
        }}
    ],

    "decision_expression": "예: FAIL OR PASS = PASS",

    "reason": "각 label의 평가 결과와 최종 연산 결과를 근거로 2~4문장으로 설명",

    "matched_evidence": [
        "최종 판단에 사용한 실제 Evidence"
    ],

    "criteria_analysis": [
        "각 label과 조건의 평가 내용"
    ]
}}
"""

    return prompt


# ============================================================
# LLM 실행
# ============================================================

def run_llm(tokenizer, model, prompt):

    messages = [
        {
            "role": "system",
            "content": "당신은 Linux 서버 보안 취약점 진단 전문가입니다."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=900,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]

    generated = outputs[0][input_length:]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return result


# ============================================================
# JSON 추출
# ============================================================

def extract_json(text):

    if not text:
        return None

    text = text.strip()

    # ```json ... ``` 제거
    text = re.sub(
        r"```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace(
        "```",
        ""
    ).strip()

    # 1차: 전체 문자열 그대로 JSON 파싱
    try:
        return json.loads(text)

    except json.JSONDecodeError:
        pass

    # 2차: 첫 { ~ 마지막 } 추출
    start = text.find("{")
    end = text.rfind("}")

    if (
        start == -1
        or end == -1
        or end <= start
    ):
        print("[DEBUG] JSON object를 찾지 못했습니다.")
        print("[DEBUG] RAW RESPONSE:")
        print(text)

        return None

    candidate = text[
        start:end + 1
    ]

    try:
        return json.loads(
            candidate
        )

    except json.JSONDecodeError as e:

        print()
        print("[DEBUG] JSON Parse Error")
        print("Line   :", e.lineno)
        print("Column :", e.colno)
        print("Reason :", e.msg)

        print()
        print("[DEBUG] RAW AI RESPONSE")
        print(text)

        print()
        print("[DEBUG] JSON CANDIDATE")
        print(candidate)

        return None


# ============================================================
# AI 결과 Validation
# ============================================================

def validate_result(result):

    valid_results = {
        "GOOD",
        "VULNERABLE",
        "MANUAL"
    }

    ai_result = str(
        result.get("result", "MANUAL")
    ).upper()

    if ai_result not in valid_results:
        ai_result = "MANUAL"


    # ============================================================
    # AI가 생성한 판단 근거 구조 확인
    # ============================================================

    label_results = result.get(
        "label_results",
        []
    )

    decision_expression = result.get(
        "decision_expression",
        ""
    )

    matched_evidence = result.get(
        "matched_evidence",
        []
    )

    criteria_analysis = result.get(
        "criteria_analysis",
        []
    )


    # ============================================================
    # GOOD / VULNERABLE 판정인데 근거가 부족하면 MANUAL 처리
    # ============================================================

    if ai_result in (
        "GOOD",
        "VULNERABLE"
    ):

        # label별 분석 결과가 없는 경우
        if not label_results:

            ai_result = "MANUAL"

            result["reason"] = (
                "AI가 GOOD/VULNERABLE 판정을 반환했지만 "
                "label별 판단 결과가 없어 "
                "MANUAL로 변경했습니다."
            )

        # OR / AND 계산 결과가 없는 경우
        elif not decision_expression:

            ai_result = "MANUAL"

            result["reason"] = (
                "AI가 GOOD/VULNERABLE 판정을 반환했지만 "
                "최종 논리 연산 결과가 없어 "
                "MANUAL로 변경했습니다."
            )

        # 실제 Evidence가 없는 경우
        elif not matched_evidence:

            ai_result = "MANUAL"

            result["reason"] = (
                "AI가 GOOD/VULNERABLE 판정을 반환했지만 "
                "판단에 사용된 실제 Evidence가 없어 "
                "MANUAL로 변경했습니다."
            )

        # 기준 비교 내용이 없는 경우
        elif not criteria_analysis:

            ai_result = "MANUAL"

            result["reason"] = (
                "AI가 GOOD/VULNERABLE 판정을 반환했지만 "
                "Evidence와 기준의 비교 결과가 없어 "
                "MANUAL로 변경했습니다."
            )


    # 최종 result 저장
    result["result"] = ai_result


    # ============================================================
    # confidence 검증
    # ============================================================

    try:
        confidence = float(
            result.get(
                "confidence",
                0.0
            )
        )

    except (
        TypeError,
        ValueError
    ):
        confidence = 0.0


    result["confidence"] = max(
        0.0,
        min(
            1.0,
            confidence
        )
    )


    # ============================================================
    # reason 검증
    # ============================================================

    reason = result.get(
        "reason"
    )

    if not reason:

        reason = (
            "AI 판단 근거가 생성되지 않았습니다."
        )

    result["reason"] = reason


    return result

# ============================================================
# Python 기반 최종 판정
# ============================================================

def calculate_final_result(
    label_results,
    knowledge_item
):

    decision_logic = knowledge_item.get(
        "decision_logic",
        {}
    )

    operator = str(
        decision_logic.get(
            "operator",
            ""
        )
    ).upper()

    good_criteria = decision_logic.get(
        "good_criteria",
        []
    )

    # Knowledge의 good_criteria label만 허용
    allowed_labels = {
        item.get("label")
        for item in good_criteria
        if item.get("label")
    }

    statuses = []

    for item in label_results:

        label = item.get("label")

        status = str(
            item.get(
                "status",
                "UNKNOWN"
            )
        ).upper()

        # good_criteria가 아닌 label은
        # 최종 연산에서 제외
        if label not in allowed_labels:
            continue

        if status not in {
            "PASS",
            "FAIL",
            "UNKNOWN"
        }:
            status = "UNKNOWN"

        statuses.append(
            status
        )

    # 평가 가능한 label 자체가 없는 경우
    if not statuses:
        return "MANUAL"

    # OR 조건
    if operator == "OR":

        if "PASS" in statuses:
            return "GOOD"

        if "UNKNOWN" in statuses:
            return "MANUAL"

        return "VULNERABLE"

    # AND 조건
    if operator == "AND":

        if all(
            status == "PASS"
            for status in statuses
        ):
            return "GOOD"

        if "UNKNOWN" in statuses:
            return "MANUAL"

        return "VULNERABLE"

    return "MANUAL"

# ============================================================
# AI 평가
# ============================================================

def evaluate_with_ai(
    code,
    evidence,
    knowledge,
    tokenizer,
    model,
    trigger_reason="NONE"
):

    kb = retrieve_knowledge(
        code,
        knowledge
    )

    # Knowledge가 없는 경우
    if kb is None:
        return {
            "result": "MANUAL",
            "confidence": 0.0,
            "reason": f"{code}에 대한 지식베이스가 없습니다.",
            "reference": []
        }

    # ------------------------------------------------------------
    # Prompt 생성
    # ------------------------------------------------------------

    prompt = build_prompt(
        code=code,
        evidence=evidence,
        knowledge_item=kb,
        trigger_reason=trigger_reason
    )

    # ------------------------------------------------------------
    # LLM 실행
    # ------------------------------------------------------------

    raw_response = run_llm(
        tokenizer,
        model,
        prompt
    )

    print()
    print("=" * 80)
    print("[DEBUG] RAW AI RESPONSE")
    print("=" * 80)
    print(raw_response)
    print("=" * 80)
    print()

    # ------------------------------------------------------------
    # JSON 추출
    # ------------------------------------------------------------

    parsed = extract_json(
        raw_response
    )

    if parsed is None:
        return {
            "result": "MANUAL",
            "confidence": 0.0,
            "reason": "AI 응답을 JSON으로 해석하지 못했습니다.",
            "reference": kb.get(
                "reference",
                []
            )
        }

    # ------------------------------------------------------------
    # 기본 Validation
    # ------------------------------------------------------------

    parsed = validate_result(
        parsed
    )

    # ============================================================
    # Knowledge의 good_criteria label 추출
    # ============================================================

    decision_logic = kb.get(
        "decision_logic",
        {}
    )

    good_criteria = decision_logic.get(
        "good_criteria",
        []
    )

    allowed_labels = {
        item.get("label")
        for item in good_criteria
        if item.get("label")
    }

    # ============================================================
    # AI label_results 필터링
    # good_criteria에 정의된 label만 최종 판정에 사용
    # ============================================================

    original_label_results = parsed.get(
        "label_results",
        []
    )

    filtered_label_results = [
    item
    for item in original_label_results
    if item.get("label") in allowed_labels
]

    # ============================================================
    # LLM이 평가하지 않은 good_criteria label은 UNKNOWN으로 추가
    # ============================================================

    existing_labels = {
        item.get("label")
        for item in filtered_label_results
    }

    for criterion in good_criteria:

        label = criterion.get("label")

        if (
            label
            and label not in existing_labels
        ):
            filtered_label_results.append(
                {
                    "label": label,
                    "status": "UNKNOWN",
                    "evidence": []
                }
            )

    parsed["label_results"] = (
        filtered_label_results
    )

    # ============================================================
    # Python 기반 최종 판정
    # ============================================================

    python_result = calculate_final_result(
        label_results=filtered_label_results,
        knowledge_item=kb
    )

    # LLM이 원래 내린 결과 저장
    parsed["llm_result"] = parsed.get(
        "result",
        "MANUAL"
    )

    # 최종 결과는 Python 계산값
    parsed["result"] = python_result

    # ============================================================
    # PASS / FAIL / UNKNOWN label 분리
    # ============================================================

    pass_labels = [
        item.get("label")
        for item in filtered_label_results
        if str(
            item.get(
                "status",
                ""
            )
        ).upper() == "PASS"
    ]

    fail_labels = [
        item.get("label")
        for item in filtered_label_results
        if str(
            item.get(
                "status",
                ""
            )
        ).upper() == "FAIL"
    ]

    unknown_labels = [
        item.get("label")
        for item in filtered_label_results
        if str(
            item.get(
                "status",
                ""
            )
        ).upper() == "UNKNOWN"
    ]

    # ============================================================
    # decision_expression Python에서 다시 생성
    # ============================================================

    operator = str(
        decision_logic.get(
            "operator",
            ""
        )
    ).upper()

    expression_parts = []

    for item in filtered_label_results:

        label = item.get(
            "label",
            "UNKNOWN"
        )

        status = str(
            item.get(
                "status",
                "UNKNOWN"
            )
        ).upper()

        expression_parts.append(
            f"{label}({status})"
        )

    if expression_parts:

        parsed["decision_expression"] = (
            f" {operator} ".join(
                expression_parts
            )
            + f" = {python_result}"
        )

    else:

        parsed["decision_expression"] = (
            "판단 가능한 good_criteria 없음"
        )

    # ============================================================
    # 최종 reason도 Python 판정에 맞춰 재생성
    # ============================================================

    if python_result == "GOOD":

        if pass_labels:

            parsed["reason"] = (
                f"{', '.join(pass_labels)} 통제가 "
                f"PASS로 확인되었습니다. "
                f"해당 항목의 good_criteria는 "
                f"{operator} 관계이므로 "
                f"최종 결과를 GOOD으로 판단합니다."
            )

        else:

            parsed["reason"] = (
                "양호 조건이 충족되어 "
                "최종 결과를 GOOD으로 판단합니다."
            )

    elif python_result == "VULNERABLE":

        fail_text = (
            ", ".join(fail_labels)
            if fail_labels
            else "확인되지 않음"
        )

        parsed["reason"] = (
            "good_criteria에 해당하는 통제 중 "
            "PASS가 확인되지 않았으며, "
            f"FAIL 항목은 {fail_text}입니다. "
            "최종 결과를 VULNERABLE로 판단합니다."
        )

    else:

        unknown_text = (
            ", ".join(unknown_labels)
            if unknown_labels
            else "확인되지 않음"
        )

        parsed["reason"] = (
            "good_criteria의 판단 결과를 "
            "확정하기 어렵습니다. "
            f"UNKNOWN 항목은 {unknown_text}입니다. "
            "최종 결과를 MANUAL로 판단합니다."
        )

    # ============================================================
    # Reference
    # ============================================================

    parsed["reference"] = kb.get(
        "reference",
        []
    )

    return parsed


# ============================================================
# 진행률 출력
# ============================================================

def print_progress(
    current,
    total,
    code=""
):

    if total <= 0:
        percent = 100.0
    else:
        percent = (
            current / total
        ) * 100

    bar_length = 30

    filled = int(
        bar_length
        * percent
        / 100
    )

    bar = (
        "#" * filled
        + "-" * (bar_length - filled)
    )

    print(
        f"\r[AI] [{bar}] "
        f"{percent:6.2f}% "
        f"({current}/{total}) "
        f"{code}",
        end="",
        flush=True
    )

    if current == total:
        print()


# ============================================================
# REVIEW=true 항목 일괄 AI 평가
# ============================================================

def evaluate_ai_items(
    items,
    knowledge,
    tokenizer,
    model
):

    """
    items 예:

    [
        {
            "code": "U-01",
            "criteria": XML Element,
            "evidence": [...]
        }
    ]
    """

    targets = []

    # REVIEW=true 항목만 추출
    for item in items:

        criteria = item["criteria"]

        ai_config = get_ai_config(
            criteria
        )

        if ai_config["review"]:

            item_copy = dict(item)

            item_copy["ai_config"] = ai_config

            targets.append(
                item_copy
            )

    total = len(targets)

    print(
        f"[+] AI REVIEW 대상 : "
        f"{total}개"
    )

    if total == 0:
        return []

    results = []

    for index, item in enumerate(
        targets,
        start=1
    ):

        code = item["code"]

        print_progress(
            index - 1,
            total,
            code
        )

        ai_config = item[
            "ai_config"
        ]

        result = evaluate_with_ai(
            code=code,
            evidence=item["evidence"],
            knowledge=knowledge,
            tokenizer=tokenizer,
            model=model,
            trigger_reason=ai_config[
                "trigger_reason"
            ]
        )

        result["code"] = code

        result["source"] = "AI"

        result["recommended"] = (
            ai_config["recommended"]
        )

        result["trigger_reason"] = (
            ai_config["trigger_reason"]
        )

        results.append(
            result
        )

        print_progress(
            index,
            total,
            code
        )

    return results

# ============================================================
# Main
# ============================================================

## 6. AI 분석 실행

In [54]:
# Notebook 환경의 경로를 기존 코드 설정에 반영
KNOWLEDGE_DIR = Path(KNOWLEDGE_DIR)

tree = ET.parse(RESULT_FILE)
root = tree.getroot()

knowledge = load_knowledge(KNOWLEDGE_DIR)

review_items = []

for item in root.findall(".//ITEM"):

    review = (
        item.findtext(
            "./AI/REVIEW",
            "false"
        )
        .strip()
        .lower()
        == "true"
    )

    if not review:
        continue

    code_value = item.get("code", "").strip()

    if not code_value:
        print("[!] code가 없는 ITEM을 건너뜁니다.")
        continue

    recommended = (
        item.findtext(
            "./AI/RECOMMENDED",
            "false"
        )
        .strip()
        .lower()
        == "true"
    )

    trigger_reason = (
        item.findtext(
            "./AI/TRIGGER_REASON",
            "NONE"
        )
        .strip()
    )

    evidence = []

    for field in item.findall("./EVIDENCE/FIELD"):

        key = field.get("key", "").strip()
        value = (field.text or "").strip()
        source = field.get("source", "").strip()
        note = field.get("note", "").strip()

        line = f"{key} = {value}"

        if source:
            line += f" [source: {source}]"

        if note:
            line += f" [note: {note}]"

        evidence.append(line)

    review_items.append(
        {
            "code": code_value,
            "evidence": evidence,
            "ai_config": {
                "review": review,
                "recommended": recommended,
                "trigger_reason": trigger_reason
            }
        }
    )

print(f"[+] AI REVIEW 대상 : {len(review_items)}개")

if not review_items:
    print("[+] AI 분석 대상이 없습니다.")
else:
    tokenizer, model = load_model()

    total = len(review_items)
    results = []

    for index, item in enumerate(review_items, start=1):

        code_value = item["code"]

        print_progress(
            index - 1,
            total,
            code_value
        )

        ai_result = evaluate_with_ai(
            code=code_value,
            evidence=item["evidence"],
            knowledge=knowledge,
            tokenizer=tokenizer,
            model=model,
            trigger_reason=item["ai_config"]["trigger_reason"]
        )

        ai_result["code"] = code_value
        ai_result["source"] = "AI"

        results.append(ai_result)

        print_progress(
            index,
            total,
            code_value
        )

    print()
    print("=" * 60)
    print("AI Analysis Result")
    print("=" * 60)

    for result in results:
        print(
            json.dumps(
                result,
                ensure_ascii=False,
                indent=2
            )
        )

[+] Knowledge Directory : knowledge
[+] Knowledge Files     : 1
[+] Loading Knowledge [1/1] 100.00% - knowledge.json
[+] Loaded Knowledge Items : 1
[+] AI REVIEW 대상 : 1개
[+] Loading Local LLM...
[+] Model : Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[+] GPU : Tesla T4
[+] Local LLM Loaded
[AI] [------------------------------]   0.00% (0/1) U-001
[DEBUG] RAW AI RESPONSE
```json
{
    "result": "VULNERABLE",
    "confidence": 0.9,

    "label_results": [
        {
            "label": "OTHER_EXECUTE_ENABLED",
            "status": "FAIL",
            "evidence": [
                "su 파일의 Other 실행 권한이 존재합니다. (/bin/su 또는 /usr/bin/su 파일의 Other 영역에 실행 권한이 존재하여 일반 사용자가 su 명령어를 실행할 수 있습니다.)"
            ]
        },
        {
            "label": "PAM_WHEEL_CONTROL",
            "status": "PASS",
            "evidence": [
                "PAM pam_wheel.so 모듈이 활성화되어 wheel 또는 지정 그룹 사용자만 su 명령어를 사용할 수 있습니다. (/etc/pam.d/su에서 pam_wheel.so 모듈이 활성화되어 wheel 또는 지정 그룹 사용자만 su 명령어를 사용할 수 있습니다.)"
            ]
        }
    ],

    "decision_expression": "PAM_WHEEL_CONTROL(PASS) OR OTHER_EXECUTE_ENABLED(FAIL) = PASS",

    "reason": "PAM_WHEEL_CONTROL가 PASS하고 OTHER_EXECUTE_ENABLED가 FAIL이므로 VULNERABLE을 반환합니다.",

    "matched_evidence": [
        "pam_

## 7. 선택: AI 결과를 JSON 파일로 저장

In [ ]:
from pathlib import Path

if 'results' in globals():
    output_file = Path("/content/ai_results.json")

    output_file.write_text(
        json.dumps(
            results,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print("Saved:", output_file)
else:
    print("저장할 AI 결과가 없습니다.")

## 8. 선택: 결과 다운로드

In [ ]:
from google.colab import files

if Path("/content/ai_results.json").exists():
    files.download("/content/ai_results.json")